In [ ]:
#| default_exp test

# test

> Run dialogs' code messages through a live solveit, `nbdev-test`-style

#| export
Testing a notebook with `nbdev-test` runs its cells under execnb, outside solveit: no dialog context, no current-message tracking, no dialog kernel. For notebooks written as solveit dialogs (like dialoghelper's own), that tests a different thing than what runs in production. `solveit-test` instead asks a running solveit instance to execute a notebook exactly as the app does -- the production runloop, a fresh dialog kernel, real current-message state -- and reports the code messages whose outputs contain errors.

In [ ]:
#| export
import asyncio, sys, time
from fastcore.utils import *
from fastcore.script import call_parse
from fastcore.parallel import parallel_async,num_cpus
from dialoghelper.core import *

In [ ]:
from fastcore.test import *

## Running one dialog

`test_dlg` is the whole protocol: restart the dialog's kernel (starting the dialog if needed, so every run begins from clean state), queue all its code messages in order -- solveit's own run-all semantics, including messages hidden from the AI -- then poll each queued message's `run` flag until the queue drains, and return the messages that ended up with error outputs. The run happens server-side, so each message executes with the dialog itself as context, exactly as when its cells were authored.

In [ ]:
#| export
async def test_dlg(
    dname:str, # Name/path of the dialog (relative to current dialog's folder, or absolute if starts with '/')
    ids:list=None, # Message ids to run (default: all code messages, including those hidden from the AI)
    timeout:int=600, # Max seconds to wait for the run to finish
    poll:float=0.5, # Seconds between completion checks
    save:bool=False, # Empty stored outputs of the messages to run first, so results reflect only this run?
)->list[dict]: # Run messages left with error outputs (empty means the dialog passed)
    "Restart `dname`'s kernel, run its code messages through the solveit runloop, and return those that errored"
    await restart_dialog(dname)
    if ids is None:
        msgs = await find_msgs(msg_type='code', context=0, include_output=False, include_meta=False, include_skipped=True, dname=dname)
        ids = [m.id for m in msgs]
    if not ids: return []
    if save: await cells_client(dname).apply([dict(op='update', id=i, outputs=[]) for i in ids])
    await run_msg(','.join(ids), dname=dname)
    end = time.time()+timeout
    for i in ids:
        while (await read_msgid(i, dname=dname)).get('run', False):
            if time.time()>end: raise TimeoutError(f'{dname}: still running after {timeout}s, at message {i}')
            await asyncio.sleep(poll)
    errs = await find_msgs(msg_type='code', only_err=True, context=0, include_skipped=True, dname=dname)
    return [m for m in errs if m.id in set(ids)]

To see it work we need a dialog on the live server, so we create a throwaway one with three code messages, the middle one broken. Note that the run continues past the failure -- solveit's run queue records the error and keeps going, so one bad cell doesn't hide later ones:

In [ ]:
tnm = f'/tmp_soltest_{os.urandom(4).hex()}'
await create_or_run_dialog(tnm, template=False)
for src in ('a=1', '1/0', 'print(a)'): await add_msg(src, msg_type='code', dname=tnm, placement='at_end')
errs = await test_dlg(tnm)
[m.id for m in errs]

KeyboardInterrupt: 

Exactly one message failed, and its output carries the traceback:

In [ ]:
test_eq(len(errs), 1)
assert 'ZeroDivisionError' in str(errs[0].output)
errs[0].output

A dialog whose run leaves no error outputs passes. `save=True` empties the stored outputs of the messages about to run, so nothing from an earlier run can masquerade as this one's result. And the scratch dialog cleans up fully -- kernel stopped, file removed:

In [ ]:
await del_msgs(errs[0].id, dname=tnm)
test_eq(await test_dlg(tnm, save=True), [])
await stop_dialog(tnm)
await rm_dialog(tnm)

## The command line

#| export
The `solveit-test` entrypoint maps each notebook path to its dialog name through the server's own base path (so the notebooks must live under the tree the instance serves), runs each in turn, and reports `nbdev-test`-style. Which messages run is aidialog's `Dialog.select_msgs`, the same selection `Dialog.execute` uses: code messages only (a prompt never runs), honoring the eval cascade — `skip_exec: true` frontmatter skips a whole notebook, `#|eval: false` (comment or meta form) skips a cell, and `nbdev_export` cells never run. A dialog that's open in solveit gets its kernel restarted by its test run, so state you had in that session is lost -- that's the price of every run starting clean. Runs execute the notebooks' code for real, mutating messages and outputs on disk, so run it on a clean checkout and review the diff: an example that doesn't clean up after itself shows up there, which is itself worth knowing. Since only messages that execute write outputs, a message skipped mid-run (a crashed kernel, a queue hiccup) keeps its stored output, and the error report can't tell it from a fresh failure. `--save` (like `nbdev-test`'s) empties the stored outputs of the messages about to run first, so outputs and report reflect this run alone.

In [ ]:
#| export
def _runnable_ids(path):
    "Message ids a non-interactive run executes, or None when the whole dialog is skipped"
    from nbdev.frontmatter import nb_frontmatter
    from aidialog.ipynb import read_ipynb
    d = read_ipynb(path)
    if d is None: return None
    if str2bool(nb_frontmatter(d).get('skip_exec', False)): return None
    return [m.id for m in d.select_msgs(all=True)]

In [ ]:
#| export
async def test_nbs(
    path:str='.', # An .ipynb file, or a directory of them, to test
    timeout:int=600, # Max seconds to wait per dialog
    keep:bool=False, # Leave dialog kernels running after their test?
    n_workers:int=None, # Max dialogs tested concurrently (default: min(num_cpus(), 8))
    save:bool=False, # Empty stored outputs of the messages to run first, so results reflect only this run?
)->dict: # Failures per notebook name: erroring message ids, or a repr'd exception
    "Test each notebook under `path` as a dialog on the local solveit instance, printing progress `nbdev-test`-style"
    root = data_root()
    p = Path(path)
    files = sorted(p.glob('*.ipynb')) if p.is_dir() else [p]
    if n_workers is None: n_workers = min(num_cpus(), 8)
    failed = {}
    async def _one(f):
        ids = _runnable_ids(f)
        if ids is None: return print(f'skip: {f.name}')
        dname = '/'+str(f.resolve().relative_to(root).with_suffix(''))
        try:
            errs = await test_dlg(dname, ids=ids, timeout=timeout, save=save)
            if errs: failed[f.name] = [m.id for m in errs]
        except Exception as e: failed[f.name] = [repr(e)]
        finally:
            if not keep: await stop_dialog(dname)
        print(f"{'FAIL' if f.name in failed else 'ok'}: {f.name}")
    await parallel_async(_one, files, n_workers=n_workers)
    return failed

In [ ]:
#| export
@call_parse(pos=['path'])
async def solveit_test(
    path:str='.', # An .ipynb file, or a directory of them, to test
    timeout:int=600, # Max seconds to wait per dialog
    keep:bool=False, # Leave dialog kernels running after their test?
    n_workers:int=None, # Max dialogs tested concurrently (default: min(num_cpus(), 8))
    save:bool=False, # Empty stored outputs of the messages to run first, so results reflect only this run?
):
    "Run notebooks as dialogs on the local solveit instance, `nbdev-test`-style; they must live under its data path"
    failed = await test_nbs(path, timeout=timeout, keep=keep, n_workers=n_workers, save=save)
    if failed:
        sys.stderr.write('\nsolveit-test failed on:\n'+'\n'.join(f'\t{k}: {", ".join(v)}' for k,v in failed.items())+'\n')
        sys.exit(1)
    print('Success.')

In [ ]:
#|hide
from nbdev import nbdev_export
nbdev_export()